In [1]:
%%capture
!pip install facenet-pytorch

In [2]:
import sys
sys.path.append('/home/pj00/projects/Github/small_face_recognition_trcking/utils')

## Libraries

In [13]:
from facenet_pytorch import MTCNN, InceptionResnetV1, fixed_image_standardization, training

import torch
import torch.nn as nn
import torch.optim as optim

from torchvision import models
from torchvision import transforms
from torchsummary import summary

from PIL import Image

import numpy as np
np.bool = np.bool_

import mxnet as mx
from mxnet import recordio

from image_iter import FaceDataset
from custom_model import distill_model
from utils import model_size

from tqdm import tqdm

## Functions

In [ ]:
def save_model(model, model_name, optimizer_name, epoch):
    os.makedirs("/kaggle/working/ckpt", exist_ok=True)  # Create ckpt directory if it doesn't exist
    file_name = f"{model_name}_{optimizer_name}_{epoch}.pt"
    save_path = os.path.join("ckpt", file_name)
    torch.save(model.state_dict(), save_path)
    print(f"Model saved to {save_path}")

## Models

In [4]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(device)

cuda:0


In [8]:
# Load model
'''
The cropped faces are passed as input in the CNN and we get an embedding for each face.
Important to set the model at .eval()
'''
teacher = InceptionResnetV1(
    classify=True,
    pretrained='casia-webface').to(device)

num_classes = teacher.logits.out_features
model_size(teacher)

model size: 925043168 / bit | 115.63 / MB


In [9]:
student = distill_model(num_classes)
student.to(device)
model_size(student)

model size: 395431392 / bit | 49.43 / MB


## Load Data

In [10]:
BATCH_SIZE=32
train_root = '/home/pj00/projects/Github/small_face_recognition_trcking/Data/CASIA/casia-webface/train.rec'

In [11]:
dataset = FaceDataset(path_imgrec=train_root, rand_mirror=True)
train_loader = torch.utils.data.DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)

/home/pj00/projects/Github/small_face_recognition_trcking/Data/CASIA/casia-webface/train.rec /home/pj00/projects/Github/small_face_recognition_trcking/Data/CASIA/casia-webface/train.idx
header0 label [490624. 501196.]
id2range 10572


## Training

In [ ]:
criterion = nn.MSELoss()
optimizer = optim.Adam(student.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=5, factor=0.5, verbose=True)

In [ ]:
BATCH_SIZE=32
epochs=100

In [ ]:
teacher.eval()
student.train()

losses = []

for epoch in tqdm(range(epochs)):
    epoch_loss = 0.0
    for images, _ in train_loader:
        images = images.to(device, dtype=torch.float32)
        
        with torch.no_grad():
            embed_teacher = teacher(images.to(device, dtype=torch.float32))
        embed_student = student(images.to(device, dtype=torch.float32))
        
        loss = criterion(embed_teacher, embed_student)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
    
    epoch_loss /= len(train_loader)
    losses.append(epoch_loss)
    print('Epoch_loss: {}'.format(epoch_loss))
    
    scheduler.step(epoch_loss)
    
    save_model(student, model_name='mobV3', optimizer_name='adam', epoch=str(epoch+1))